# LeetCode 3562 — Maximum Profit from Trading Stocks with Discounts bằng Python

- Link: https://leetcode.com/problems/maximum-profit-from-trading-stocks-with-discounts/
- Độ khó: **Hard**
- Chủ đề: **Quy hoạch động trên cây** + **knapsack (balo) theo ngân sách**, trạng thái "cha đã mua hay chưa", **gộp balo các con** (max-plus convolution)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxProfit(self, n: int, present: List[int], future: List[int], hierarchy: List[List[int]], budget: int) -> int:
```

## 1. Mục tiêu học

1. Nhận ra bài là **balo 0/1 trên cây**: mỗi nhân viên là một món đồ (mua hoặc không), chi phí phụ thuộc việc **sếp trực tiếp** có mua hay không.
2. Thiết kế trạng thái `dp[u][p][b]`: lợi nhuận lớn nhất trong cây con `u`, biết **cha của `u` đã mua (`p = 1`) hay chưa (`p = 0`)**, với tổng chi ≤ `b`.
3. Viết phép **gộp balo hai con** (max-plus convolution) `new[b] = max(A[a] + C[b − a])`.
4. Tối ưu bằng cách giới hạn ngân sách theo **tổng chi phí tối đa của cây con** (không duyệt các `b` vô nghĩa).
5. Kiểm chứng bằng vét cạn mọi tập người mua trên cây nhỏ.

## 2. Đề bài & ví dụ

Công ty có `n` nhân viên (ID `1..n`), nhân viên 1 là CEO. `hierarchy[i] = [u, v]`: `u` là **sếp trực tiếp** của `v` (cây gốc 1).

- `present[i]`: giá mua cổ phiếu hôm nay của nhân viên `i+1`; `future[i]`: giá bán ngày mai.
- **Giảm giá:** nếu sếp trực tiếp của `v` mua cổ phiếu **của sếp**, thì `v` mua với giá `floor(present[v] / 2)`.
- Mỗi cổ phiếu mua tối đa một lần; tổng tiền mua ≤ `budget` (không dùng lợi nhuận để mua thêm).

Trả về **lợi nhuận lớn nhất** (`Σ (future − giá mua)` trên những người được mua).

**Ví dụ 1:** `n=2, present=[1,2], future=[4,3], hierarchy=[[1,2]], budget=3` → `5`.

**Ví dụ 2:** `n=2, present=[3,4], future=[5,8], hierarchy=[[1,2]], budget=4` → `4`.

**Ví dụ 3:** `n=3, present=[4,6,8], future=[7,9,11], hierarchy=[[1,2],[1,3]], budget=10` → `10`.

**Ví dụ 4:** `n=3, present=[5,2,3], future=[8,5,6], hierarchy=[[1,2],[2,3]], budget=7` → `12`.

**Ràng buộc:** `1 ≤ n ≤ 160`, `1 ≤ present, future ≤ 50`, `1 ≤ budget ≤ 160`.

In [1]:
from typing import List
examples = [
    (2, [1, 2], [4, 3], [[1, 2]], 3, 5),
    (2, [3, 4], [5, 8], [[1, 2]], 4, 4),
    (3, [4, 6, 8], [7, 9, 11], [[1, 2], [1, 3]], 10, 10),
    (3, [5, 2, 3], [8, 5, 6], [[1, 2], [2, 3]], 7, 12),
]
for n, pr, fu, h, b, exp in examples:
    print(f"n={n}, present={pr}, future={fu}, hierarchy={h}, budget={b} -> kỳ vọng {exp}")

n=2, present=[1, 2], future=[4, 3], hierarchy=[[1, 2]], budget=3 -> kỳ vọng 5
n=2, present=[3, 4], future=[5, 8], hierarchy=[[1, 2]], budget=4 -> kỳ vọng 4
n=3, present=[4, 6, 8], future=[7, 9, 11], hierarchy=[[1, 2], [1, 3]], budget=10 -> kỳ vọng 10
n=3, present=[5, 2, 3], future=[8, 5, 6], hierarchy=[[1, 2], [2, 3]], budget=7 -> kỳ vọng 12


## 3. Kiến thức nền

### 3.1. Dựng cây từ danh sách cạnh "sếp → nhân viên"

Chuyển ID về 0-based (`u − 1`), dựng `children[u]`. Duyệt BFS từ gốc 0 để có thứ tự cha trước con; đi ngược thứ tự để xử lý con trước cha.

In [2]:
def build_tree(n, hierarchy):
    children = [[] for _ in range(n)]
    for u, v in hierarchy:
        children[u - 1].append(v - 1)
    order = [0]
    for u in order:
        order.extend(children[u])
    return children, order

print(build_tree(3, [[1, 2], [2, 3]]))

([[1], [2], []], [0, 1, 2])


### 3.2. Balo 0/1 với ngân sách

`best[b]` = lợi nhuận lớn nhất với tổng chi **≤ b**. Thêm món (giá `c`, lãi `g`): duyệt `b` **giảm dần** `best[b] = max(best[b], best[b − c] + g)`.

In [3]:
def knapsack(items, B):
    best = [0] * (B + 1)
    for c, g in items:
        for b in range(B, c - 1, -1):
            best[b] = max(best[b], best[b - c] + g)
    return best

print(knapsack([(1, 3), (2, 1), (2, 4)], 4))

[0, 3, 4, 7, 7]


### 3.3. Gộp hai bảng balo (max-plus convolution)

Hai nhóm độc lập có bảng `A`, `C` (ngân sách ≤ b). Bảng gộp: `new[b] = max_{0 ≤ a ≤ b} A[a] + C[b − a]` — chia ngân sách cho hai nhóm. Chi phí `O(B^2)`.

In [4]:
def merge(A, C, B):
    new = [0] * (B + 1)
    for a in range(B + 1):
        if A[a] == A[a - 1] and a > 0:
            continue                         # A[a] không hơn A[a-1]: bỏ qua (tối ưu nhỏ)
        Aa = A[a]
        for c in range(B - a + 1):
            v = Aa + C[c]
            if v > new[a + c]:
                new[a + c] = v
    for b in range(1, B + 1):                # đảm bảo tính "≤ b" (không giảm)
        if new[b] < new[b - 1]:
            new[b] = new[b - 1]
    return new

A = knapsack([(1, 3)], 4)
C = knapsack([(2, 4), (1, 1)], 4)
print('A =', A, '| C =', C, '| gộp =', merge(A, C, 4), '| balo chung =', knapsack([(1, 3), (2, 4), (1, 1)], 4))

A = [0, 3, 3, 3, 3] | C = [0, 1, 4, 5, 5] | gộp = [0, 3, 4, 7, 8] | balo chung = [0, 3, 4, 7, 8]


### 3.4. Phép chia lấy phần nguyên

`floor(present / 2)` trong Python là `present // 2` (số dương). Ví dụ `3 // 2 = 1`, `1 // 2 = 0` — giá giảm có thể bằng **0**.

In [5]:
print([p // 2 for p in [1, 2, 3, 8, 50]])

[0, 1, 1, 4, 25]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Quyết định của mỗi nhân viên chỉ ảnh hưởng tới **các con trực tiếp** (giá giảm). Vậy khi giải cây con của `u`, thông tin cần biết từ bên ngoài chỉ là: **cha của `u` có mua không** (quyết định giá của `u`) và **ngân sách** được cấp cho cây con này.

Tính tay ví dụ 4 (đường thẳng 1 → 2 → 3, budget 7):

| tập mua | chi phí | lợi nhuận |
|---|---|---|
| {1, 2, 3} | 5 + 2//2 + 3//2 = 5 + 1 + 1 = 7 | (8−5) + (5−1) + (6−1) = 12 |
| {2, 3} | 2 + 1 = 3 | 3 + 5 = 8 |
| {1, 3} | 5 + 3 = 8 > 7 | — |

Đáp án `12`: mua cả ba — nhờ dây chuyền giảm giá.

In [6]:
def cost_profit(n, present, future, hierarchy, chosen):
    boss = [-1] * n
    for u, v in hierarchy:
        boss[v - 1] = u - 1
    cost = profit = 0
    for v in chosen:
        price = present[v] // 2 if boss[v] in chosen else present[v]
        cost += price
        profit += future[v] - price
    return cost, profit

for chosen in [{0, 1, 2}, {1, 2}, {0, 2}]:
    print(sorted(c + 1 for c in chosen), cost_profit(3, [5, 2, 3], [8, 5, 6], [[1, 2], [2, 3]], chosen))

[1, 2, 3] (7, 12)
[2, 3] (3, 8)
[1, 3] (8, 6)


## 5. Cách 1 — Brute Force: thử mọi tập người mua

`2^n` tập; với mỗi tập tính tổng chi (có giảm giá) và lợi nhuận, giữ max trong các tập vừa ngân sách.

In [7]:
def max_profit_brute(n, present, future, hierarchy, budget):
    best = 0
    for mask in range(1 << n):
        chosen = {i for i in range(n) if mask >> i & 1}
        c, p = cost_profit(n, present, future, hierarchy, chosen)
        if c <= budget:
            best = max(best, p)
    return best

for n, pr, fu, h, b, exp in examples:
    print(max_profit_brute(n, pr, fu, h, b), '| kỳ vọng', exp)

5 | kỳ vọng 5
4 | kỳ vọng 4
10 | kỳ vọng 10
12 | kỳ vọng 12


**Độ phức tạp:** `O(2^n · n)` — chỉ dùng cho `n ≤ 15`.

**Nút thắt:** các quyết định ở các nhánh khác nhau của cây **độc lập** (chỉ liên hệ qua cha). DP trên cây khai thác điều này: giải từng cây con với "trạng thái cha" và "ngân sách" như tham số.

## 6. Ý tưởng thuật toán / bước đột phá

Định nghĩa `dp[u][p]` là **mảng** độ dài `B + 1`: `dp[u][p][b]` = lợi nhuận lớn nhất trong cây con `u` với tổng chi ≤ `b`, biết cha của `u` **đã mua** (`p = 1`) hay **chưa** (`p = 0`).

Với mỗi `u` có hai lựa chọn:

- **Không mua `u`:** mọi con `c` thấy "cha chưa mua" ⇒ gộp các bảng `dp[c][0]` ⇒ bảng `skip`.
- **Mua `u`:** mọi con thấy "cha đã mua" ⇒ gộp các bảng `dp[c][1]` ⇒ bảng `take`. Giá của `u` là `cost = present[u]` nếu `p = 0`, `present[u] // 2` nếu `p = 1`; lãi `future[u] − cost`. Khi đó `buy_p[b] = take[b − cost] + future[u] − cost` với `b ≥ cost`.

`dp[u][p][b] = max(skip[b], buy_p[b])`. Đáp án: `dp[gốc][0][budget]` (CEO không có sếp).

Quy tắc:

- **Nếu** `u` được mua **thì** các con dùng bảng "cha đã mua".
- **Nếu** `u` không mua **thì** các con dùng bảng "cha chưa mua".
- Lợi nhuận có thể âm (`future < giá`) — khi đó DP tự chọn không mua.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP trên cây với từ điển `chi phí → lợi nhuận` (tập Pareto)

Thay mảng bằng `dict` các cặp (chi phí chính xác, lợi nhuận tốt nhất), chỉ giữ chi phí ≤ budget. Trực quan, tự nhiên bỏ qua chi phí không đạt được; nhưng gộp hai dict tốn `|A|·|C|`.

In [8]:
def max_profit_dict(n, present, future, hierarchy, budget):
    children, order = build_tree(n, hierarchy)

    def combine(A, C):
        out = {}
        for a, pa in A.items():
            for c, pc in C.items():
                if a + c <= budget and out.get(a + c, -1) < pa + pc:
                    out[a + c] = pa + pc
        return out

    dp = [None] * n
    for u in reversed(order):
        skip, take = {0: 0}, {0: 0}
        for c in children[u]:
            skip = combine(skip, dp[c][0])
            take = combine(take, dp[c][1])
        res = []
        for p in (0, 1):
            cost = present[u] // 2 if p else present[u]
            d = dict(skip)
            for t, pt in take.items():
                if t + cost <= budget:
                    val = pt + future[u] - cost
                    if d.get(t + cost, -10**9) < val:
                        d[t + cost] = val
            res.append(d)
        dp[u] = res
    return max(dp[0][0].values())

for n, pr, fu, h, b, exp in examples:
    print(max_profit_dict(n, pr, fu, h, b), '| kỳ vọng', exp)

5 | kỳ vọng 5
4 | kỳ vọng 4
10 | kỳ vọng 10
12 | kỳ vọng 12


**Độ phức tạp:** mỗi dict tối đa `B + 1` khóa ⇒ mỗi lần gộp `O(B^2)`; tổng `O(n · B^2)` nhưng hằng số `dict` lớn.

### 7.2. Cách 3 — DP mảng ngân sách (lời giải chính)

Dùng mảng `[0..B]` với ngữ nghĩa "chi **≤ b**" (mảng không giảm), gộp bằng hàm `merge` ở mục 3.3.

In [9]:
def max_profit_array(n, present, future, hierarchy, budget):
    B = budget
    children, order = build_tree(n, hierarchy)
    dp = [None] * n
    for u in reversed(order):
        skip = [0] * (B + 1)
        take = [0] * (B + 1)
        for c in children[u]:
            skip = merge(skip, dp[c][0], B)
            take = merge(take, dp[c][1], B)
        res = []
        for p in (0, 1):
            cost = present[u] // 2 if p else present[u]
            gain = future[u] - cost
            row = skip[:]
            for b in range(cost, B + 1):
                v = take[b - cost] + gain
                if v > row[b]:
                    row[b] = v
            res.append(row)
        dp[u] = res
    return dp[0][0][B]

for n, pr, fu, h, b, exp in examples:
    print(max_profit_array(n, pr, fu, h, b), '| kỳ vọng', exp)

5 | kỳ vọng 5
4 | kỳ vọng 4
10 | kỳ vọng 10
12 | kỳ vọng 12


**Độ phức tạp:** mỗi cạnh cha–con gây 2 lần gộp `O(B^2)` ⇒ `O(n · B^2)` ≈ `160 · 2 · 160^2 / 2 ≈ 4·10^6` phép cộng — vài giây trong Python thuần.

### 7.3. Cách 4 — Giới hạn ngân sách theo cây con

Cây con `u` không bao giờ tiêu quá `cap[u] = min(B, Σ present trong cây con)`. Bảng của `u` chỉ cần độ dài `cap[u] + 1`, và phép gộp chỉ chạy `a ≤ cap(A)`, `c ≤ cap(C)` — giống kỹ thuật "gộp theo kích thước cây con", tổng chi phí giảm mạnh khi các cây con nhỏ.

In [10]:
def merge_capped(A, C, B):
    la, lc = len(A) - 1, len(C) - 1
    top = min(B, la + lc)
    new = [0] * (top + 1)
    for a in range(la + 1):
        Aa = A[a]
        hi = min(lc, top - a)
        for c in range(hi + 1):
            v = Aa + C[c]
            if v > new[a + c]:
                new[a + c] = v
    for b in range(1, top + 1):
        if new[b] < new[b - 1]:
            new[b] = new[b - 1]
    return new

def max_profit_capped(n, present, future, hierarchy, budget):
    B = budget
    children, order = build_tree(n, hierarchy)
    dp = [None] * n
    for u in reversed(order):
        skip, take = [0], [0]
        for c in children[u]:
            skip = merge_capped(skip, dp[c][0], B)
            take = merge_capped(take, dp[c][1], B)
        res = []
        for p in (0, 1):
            cost = present[u] // 2 if p else present[u]
            gain = future[u] - cost
            top = min(B, len(take) - 1 + cost)
            row = skip + [skip[-1]] * (top + 1 - len(skip)) if top + 1 > len(skip) else skip[:top + 1]
            for b in range(cost, top + 1):
                v = take[min(b - cost, len(take) - 1)] + gain
                if v > row[b]:
                    row[b] = v
            for b in range(1, top + 1):
                if row[b] < row[b - 1]:
                    row[b] = row[b - 1]
            res.append(row)
        dp[u] = res
    return dp[0][0][-1]

for n, pr, fu, h, b, exp in examples:
    print(max_profit_capped(n, pr, fu, h, b), '| kỳ vọng', exp)

5 | kỳ vọng 5
4 | kỳ vọng 4
10 | kỳ vọng 10
12 | kỳ vọng 12


Bảng của mỗi cây con có độ dài tối đa `min(B, Σ present) + 1` và **phần tử cuối** là lợi nhuận tốt nhất với mọi ngân sách lớn hơn (mảng không giảm), nên đọc `dp[0][0][-1]` là đúng.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 4 (đường thẳng `1 → 2 → 3`, budget 7): in hai bảng `dp[u][0]`, `dp[u][1]` của từng nhân viên từ lá lên.

In [11]:
def trace(n, present, future, hierarchy, budget):
    B = budget
    children, order = build_tree(n, hierarchy)
    dp = [None] * n
    for u in reversed(order):
        skip, take = [0] * (B + 1), [0] * (B + 1)
        for c in children[u]:
            skip = merge(skip, dp[c][0], B)
            take = merge(take, dp[c][1], B)
        res = []
        for p in (0, 1):
            cost = present[u] // 2 if p else present[u]
            gain = future[u] - cost
            row = skip[:]
            for b in range(cost, B + 1):
                row[b] = max(row[b], take[b - cost] + gain)
            res.append(row)
        dp[u] = res
        print(f"Nhân viên {u + 1}: giá gốc {present[u]}, giá giảm {present[u] // 2}, bán {future[u]}")
        print(f"   skip (không mua)     = {skip}")
        print(f"   take (con có giảm)   = {take}")
        print(f"   dp[cha chưa mua]     = {res[0]}")
        print(f"   dp[cha đã mua]       = {res[1]}")
    print('Đáp án = dp[CEO][0][budget] =', dp[0][0][B])
    return dp[0][0][B]

trace(3, [5, 2, 3], [8, 5, 6], [[1, 2], [2, 3]], 7)

Nhân viên 3: giá gốc 3, giá giảm 1, bán 6
   skip (không mua)     = [0, 0, 0, 0, 0, 0, 0, 0]
   take (con có giảm)   = [0, 0, 0, 0, 0, 0, 0, 0]
   dp[cha chưa mua]     = [0, 0, 0, 3, 3, 3, 3, 3]
   dp[cha đã mua]       = [0, 5, 5, 5, 5, 5, 5, 5]
Nhân viên 2: giá gốc 2, giá giảm 1, bán 5
   skip (không mua)     = [0, 0, 0, 3, 3, 3, 3, 3]
   take (con có giảm)   = [0, 5, 5, 5, 5, 5, 5, 5]
   dp[cha chưa mua]     = [0, 0, 3, 8, 8, 8, 8, 8]
   dp[cha đã mua]       = [0, 4, 9, 9, 9, 9, 9, 9]
Nhân viên 1: giá gốc 5, giá giảm 2, bán 8
   skip (không mua)     = [0, 0, 3, 8, 8, 8, 8, 8]
   take (con có giảm)   = [0, 4, 9, 9, 9, 9, 9, 9]
   dp[cha chưa mua]     = [0, 0, 3, 8, 8, 8, 8, 12]
   dp[cha đã mua]       = [0, 0, 6, 10, 15, 15, 15, 15]
Đáp án = dp[CEO][0][budget] = 12


12

Đọc trace: ở nhân viên 2, `dp[cha đã mua][2] = 4 + 5 = 9` (mua 2 với giá 1 và 3 với giá 1). Ở CEO, mua với giá 5 rồi còn 2 cho cây con ⇒ `3 + 9 = 12`.

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def maxProfit(self, n: int, present: List[int], future: List[int], hierarchy: List[List[int]], budget: int) -> int:
        B = budget
        children = [[] for _ in range(n)]
        for u, v in hierarchy:
            children[u - 1].append(v - 1)
        order = [0]
        for u in order:
            order.extend(children[u])

        def merge(A, C):
            la, lc = len(A) - 1, len(C) - 1
            top = min(B, la + lc)
            new = [0] * (top + 1)
            for a in range(la + 1):
                Aa = A[a]
                for c in range(min(lc, top - a) + 1):
                    v = Aa + C[c]
                    if v > new[a + c]:
                        new[a + c] = v
            for b in range(1, top + 1):
                if new[b] < new[b - 1]:
                    new[b] = new[b - 1]
            return new

        dp = [None] * n
        for u in reversed(order):
            skip, take = [0], [0]
            for c in children[u]:
                skip = merge(skip, dp[c][0])
                take = merge(take, dp[c][1])
            res = []
            for p in (0, 1):
                cost = present[u] // 2 if p else present[u]
                gain = future[u] - cost
                top = min(B, len(take) - 1 + cost)
                row = skip[:top + 1] + [skip[-1]] * max(0, top + 1 - len(skip))
                lt = len(take) - 1
                for b in range(cost, top + 1):
                    v = take[min(b - cost, lt)] + gain
                    if v > row[b]:
                        row[b] = v
                for b in range(1, top + 1):
                    if row[b] < row[b - 1]:
                        row[b] = row[b - 1]
                res.append(row)
            dp[u] = res
            for c in children[u]:
                dp[c] = None
        return dp[0][0][-1]

for n, pr, fu, h, b, exp in examples:
    got = Solution().maxProfit(n, pr, fu, h, b)
    print(got, 'OK' if got == exp else 'SAI')

5 OK
4 OK
10 OK
12 OK


## 10. Vì sao thuật toán đúng

1. **Tính độc lập:** chi phí của `v` chỉ phụ thuộc quyết định của `v` và sếp trực tiếp. Khi đã biết quyết định của `u`, các cây con của các con của `u` hoàn toàn độc lập (chỉ chia nhau ngân sách) ⇒ gộp bằng max-plus convolution là chính xác.
2. **Trạng thái đủ:** `dp[u][p]` cần biết `p` vì nó quyết định giá của `u`; mọi thông tin khác bên ngoài cây con không ảnh hưởng.
3. **Ngữ nghĩa "≤ b":** khởi tạo bằng 0 (không mua gì) và giữ mảng không giảm; phép gộp lấy max trên mọi cách chia ngân sách ⇒ `new[b]` là max trên mọi cặp chi phí có tổng ≤ b.
4. **Hai lựa chọn của `u` phủ hết khả năng:** không mua (con dùng `dp[c][0]`) hoặc mua (con dùng `dp[c][1]`, trả `cost`).
5. **Cắt ngân sách theo cây con (cách 4):** cây con không thể tiêu quá tổng giá gốc của nó; phần tử cuối của mảng không giảm đại diện cho mọi ngân sách lớn hơn ⇒ không mất nghiệm.

## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 160`, `B = budget ≤ 160`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n · n)` | `O(n)` |
| 2. DP dict | `O(n · B^2)` (hằng số dict lớn) | `O(n · B)` |
| 3. DP mảng | `O(n · B^2)` | `O(n · B)` |
| 4. DP mảng cắt theo cây con | `O(n · B^2)` xấu nhất, thường nhỏ hơn nhiều | `O(n · B)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [13]:
impls = {
    'dict': max_profit_dict,
    'array': max_profit_array,
    'capped': max_profit_capped,
    'submit': lambda *a: Solution().maxProfit(*a),
}
edge = [
    (1, [5], [3], [], 10, 0),                      # lỗ -> không mua
    (1, [5], [9], [], 4, 0),                       # không đủ tiền
    (1, [5], [9], [], 5, 4),
    (2, [1, 1], [50, 50], [[1, 2]], 1, 99),        # giá giảm 1 // 2 = 0: lãi 49 + 50
    (3, [10, 1, 1], [11, 9, 9], [[1, 2], [1, 3]], 2, 16),   # bỏ CEO, mua 2 nhân viên giá gốc
]
for n, pr, fu, h, b, exp in examples + edge:
    for name, f in impls.items():
        got = f(n, pr, fu, h, b)
        assert got == exp, (name, n, pr, fu, h, b, got, exp)
    assert max_profit_brute(n, pr, fu, h, b) == exp
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [14]:
import random
random.seed(3562)

def rand_case(n, pmax=12, bmax=30):
    hierarchy = [[random.randint(1, v - 1), v] for v in range(2, n + 1)]
    present = [random.randint(1, pmax) for _ in range(n)]
    future = [random.randint(1, pmax + 5) for _ in range(n)]
    return n, present, future, hierarchy, random.randint(1, bmax)

cnt = 0
for _ in range(400):
    case = rand_case(random.randint(1, 11))
    exp = max_profit_brute(*case)
    for name, f in impls.items():
        assert f(*case) == exp, (name, case)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cây ngẫu nhiên.')

Khớp vét cạn trên 400 cây ngẫu nhiên.


### 12.3. Ca lớn (`n = 160`, `budget = 160`, giá tới 50)

In [15]:
import time
random.seed(160)
for shape in ['ngẫu nhiên', 'đường thẳng', 'sao']:
    n = 160
    if shape == 'ngẫu nhiên':
        h = [[random.randint(1, v - 1), v] for v in range(2, n + 1)]
    elif shape == 'đường thẳng':
        h = [[v - 1, v] for v in range(2, n + 1)]
    else:
        h = [[1, v] for v in range(2, n + 1)]
    pr = [random.randint(1, 50) for _ in range(n)]
    fu = [random.randint(1, 50) for _ in range(n)]
    t = time.time(); a = Solution().maxProfit(n, pr, fu, h, 160); t1 = time.time() - t
    t = time.time(); b = max_profit_array(n, pr, fu, h, 160); t2 = time.time() - t
    assert a == b
    print(f"{shape:>11}: đáp án {a} | cắt theo cây con {t1:.2f}s | mảng đầy đủ {t2:.2f}s")

 ngẫu nhiên: đáp án 858 | cắt theo cây con 0.03s | mảng đầy đủ 0.03s
đường thẳng: đáp án 642 | cắt theo cây con 0.01s | mảng đầy đủ 0.01s


        sao: đáp án 1078 | cắt theo cây con 0.08s | mảng đầy đủ 0.19s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Giảm giá dựa trên **tổ tiên bất kỳ** đã mua (không chỉ sếp trực tiếp)

In [16]:
def wrong_any_ancestor(n, present, future, hierarchy, budget):
    boss = [-1] * n
    for u, v in hierarchy:
        boss[v - 1] = u - 1
    best = 0
    for mask in range(1 << n):
        chosen = {i for i in range(n) if mask >> i & 1}
        cost = profit = 0
        for v in chosen:
            a, disc = boss[v], False
            while a != -1:
                if a in chosen:
                    disc = True                    # SAI: ông/bà sếp cũng cho giảm
                a = boss[a]
            price = present[v] // 2 if disc else present[v]
            cost += price; profit += future[v] - price
        if cost <= budget:
            best = max(best, profit)
    return best

random.seed(11)
case = None
for _ in range(3000):
    c = rand_case(random.randint(3, 6))
    if wrong_any_ancestor(*c) != max_profit_brute(*c):
        case = c
        break
print('phản ví dụ (n, present, future, hierarchy, budget):', case)
print('sai :', wrong_any_ancestor(*case))
print('đúng:', Solution().maxProfit(*case))

phản ví dụ (n, present, future, hierarchy, budget): (3, [3, 11, 9], [7, 15, 17], [[1, 2], [2, 3]], 7)
sai : 17
đúng: 4


**Vì sao sai:** chỉ **sếp trực tiếp** mua mới cho giảm giá. Trong phản ví dụ, code sai cho một nhân viên được giảm giá nhờ "ông sếp" dù sếp trực tiếp không mua, nên lợi nhuận bị tính quá cao.

### Lỗi 2 — Dùng lợi nhuận để mua thêm (ngân sách = budget + lãi)

In [17]:
def wrong_reinvest(n, present, future, hierarchy, budget):
    boss = [-1] * n
    for u, v in hierarchy:
        boss[v - 1] = u - 1
    best = 0
    for mask in range(1 << n):
        chosen = {i for i in range(n) if mask >> i & 1}
        cost = profit = 0
        for v in chosen:
            price = present[v] // 2 if boss[v] in chosen else present[v]
            cost += price; profit += future[v] - price
        if cost <= budget + max(0, profit):        # SAI
            best = max(best, profit)
    return best

case = (2, [3, 4], [5, 8], [[1, 2]], 4)
print('sai :', wrong_reinvest(*case))     # coi như mua được cả hai
print('đúng:', Solution().maxProfit(*case))

sai : 8
đúng: 4


**Vì sao sai:** đề nói không được dùng lợi nhuận ngày mai để mua hôm nay (ví dụ 2).

### Lỗi 3 — Gộp con bằng cách **cộng từng vị trí** (`A[b] + C[b]`) thay vì chia ngân sách

In [18]:
def merge_wrong(A, C, B):
    return [A[b] + C[b] for b in range(B + 1)]      # SAI: hai con cùng dùng toàn bộ b

def max_profit_wrong3(n, present, future, hierarchy, budget):
    B = budget
    children, order = build_tree(n, hierarchy)
    dp = [None] * n
    for u in reversed(order):
        skip, take = [0] * (B + 1), [0] * (B + 1)
        for c in children[u]:
            skip = merge_wrong(skip, dp[c][0], B)
            take = merge_wrong(take, dp[c][1], B)
        res = []
        for p in (0, 1):
            cost = present[u] // 2 if p else present[u]
            row = skip[:]
            for b in range(cost, B + 1):
                row[b] = max(row[b], take[b - cost] + future[u] - cost)
            res.append(row)
        dp[u] = res
    return dp[0][0][B]

case = (3, [4, 6, 8], [7, 9, 11], [[1, 2], [1, 3]], 10)
print('sai :', max_profit_wrong3(*case))
print('đúng:', Solution().maxProfit(*case))

sai : 16
đúng: 10


**Vì sao sai:** hai cây con **chia nhau** ngân sách; cộng từng vị trí ngầm cho mỗi con tiêu tới `b` ⇒ vượt ngân sách.

### Lỗi 4 — Chỉ một bảng `dp[u]` (quên trạng thái cha đã mua)

In [19]:
def max_profit_one_state(n, present, future, hierarchy, budget):
    B = budget
    children, order = build_tree(n, hierarchy)
    dp = [None] * n
    for u in reversed(order):
        acc = [0] * (B + 1)
        for c in children[u]:
            acc = merge(acc, dp[c], B)
        cost = present[u]                            # SAI: không bao giờ được giảm giá
        row = acc[:]
        for b in range(cost, B + 1):
            row[b] = max(row[b], acc[b - cost] + future[u] - cost)
        dp[u] = row
    return dp[0][B]

case = (3, [5, 2, 3], [8, 5, 6], [[1, 2], [2, 3]], 7)
print('sai :', max_profit_one_state(*case))
print('đúng:', Solution().maxProfit(*case))

sai : 6
đúng: 12


**Vì sao sai:** giá của `u` phụ thuộc cha; thiếu chiều trạng thái đó thì không bao giờ khai thác được giảm giá (hoặc khai thác sai).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn tập nút trên cây, có **ngân sách**" ⇒ **tree knapsack**: bảng theo ngân sách cho mỗi cây con, gộp con bằng max-plus.
- "Giá/lợi ích của nút phụ thuộc nút cha" ⇒ thêm chiều trạng thái **cha đã chọn hay chưa** (giống House Robber III).
- Ngữ nghĩa "≤ b" (mảng không giảm) giúp khởi tạo bằng 0, không cần −∞.
- Cắt độ dài bảng theo khả năng tiêu của cây con ⇒ nhanh hơn nhiều (kỹ thuật "gộp theo kích thước").

## 15. Biến thể của bài

1. Giảm giá áp dụng khi **bất kỳ tổ tiên** mua ⇒ trạng thái "có tổ tiên đã mua" (vẫn 2 trạng thái, truyền xuống).
2. Giảm giá theo **số con** của sếp đã mua ⇒ phức tạp hơn.
3. Không có ngân sách ⇒ DP 2 trạng thái đơn giản, `O(n)`.
4. Bài liên quan: LC 337 *House Robber III*, LC 2858 / bài balo trên cây (tree knapsack) kinh điển.

In [20]:
# Biến thể 3: không giới hạn ngân sách -> DP O(n)
def max_profit_unlimited(n, present, future, hierarchy):
    children, order = build_tree(n, hierarchy)
    f = [[0, 0] for _ in range(n)]          # f[u][p]
    for u in reversed(order):
        skip = sum(f[c][0] for c in children[u])
        take = sum(f[c][1] for c in children[u])
        for p in (0, 1):
            cost = present[u] // 2 if p else present[u]
            f[u][p] = max(skip, take + future[u] - cost)
    return f[0][0]

random.seed(7)
for _ in range(100):
    case = rand_case(random.randint(1, 9), bmax=10**4)
    n, pr, fu, h, _ = case
    assert max_profit_unlimited(n, pr, fu, h) == max_profit_brute(n, pr, fu, h, 10**4)
print('Biến thể 3 khớp vét cạn; ví dụ:', max_profit_unlimited(3, [5, 2, 3], [8, 5, 6], [[1, 2], [2, 3]]))

Biến thể 3 khớp vét cạn; ví dụ: 12


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Vét cạn tập mua | `O(2^n n)` | `O(n)` | Oracle `n ≤ 15` |
| 2 | DP dict chi phí | `O(n B^2)` | `O(n B)` | Khi chi phí thưa |
| 3 | DP mảng ngân sách | `O(n B^2)` | `O(n B)` | Dễ viết, chuẩn |
| 4 | Mảng cắt theo cây con | ≤ `O(n B^2)` | `O(n B)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maxProfit(2, [2, 2], [3, 3], [[1, 2]], 2)` = ?

**Bài 2 (tính tay):** ví dụ 3 với `budget = 4`.

**Bài 3 (điền TODO):** viết `merge_todo(A, C, B)`.

**Bài 4 (tự viết lại):** biến thể 1 — giảm giá khi **bất kỳ tổ tiên** đã mua; viết DP và kiểm bằng vét cạn.

**Bài 5 (mini challenge):** truy vết **tập nhân viên** được mua trong lời giải tối ưu.

In [21]:
def merge_todo(A, C, B):
    # TODO: new[b] = max_{a<=b} A[a] + C[b-a]
    raise NotImplementedError

def ancestor_discount_todo(n, present, future, hierarchy, budget):
    # TODO
    raise NotImplementedError

def chosen_set_todo(n, present, future, hierarchy, budget):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mua CEO giá 2, nhân viên 2 giá 1 ⇒ tổng 3 > 2. (2) Chỉ mua một người: lãi 1. (3) Đáp án 1.

**Bài 2.** (1) Mua CEO (4) thì hết tiền: lãi 3. (2) Không mua CEO: mua 2 (6) quá tiền. (3) Đáp án 3.

**Bài 3.** (1) Hai vòng `a`, `c` với `a + c ≤ B`. (2) Giữ max. (3) Chuẩn hóa mảng không giảm.

**Bài 4.** (1) Trạng thái `p` = "có tổ tiên đã mua". (2) Khi mua `u`, con nhận `p = 1`; khi không mua, con nhận `p` của `u` (truyền tiếp). (3) Vét cạn: giảm khi có tổ tiên trong tập.

**Bài 5.** (1) Lưu lựa chọn tại mỗi `(u, p, b)` và cách chia ngân sách khi gộp. (2) Hoặc đơn giản: vét cạn nhỏ để kiểm tra. (3) Ở đây dùng truy vết bằng đệ quy với ghi nhớ cách chia.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2, 3
a1 = Solution().maxProfit(2, [2, 2], [3, 3], [[1, 2]], 2)
a2 = Solution().maxProfit(3, [4, 6, 8], [7, 9, 11], [[1, 2], [1, 3]], 4)
assert a1 == 1 == max_profit_brute(2, [2, 2], [3, 3], [[1, 2]], 2)
assert a2 == 3 == max_profit_brute(3, [4, 6, 8], [7, 9, 11], [[1, 2], [1, 3]], 4)

def merge_ans(A, C, B):
    new = [0] * (B + 1)
    for a in range(B + 1):
        for c in range(B - a + 1):
            new[a + c] = max(new[a + c], A[a] + C[c])
    for b in range(1, B + 1):
        new[b] = max(new[b], new[b - 1])
    return new

random.seed(31)
for _ in range(200):
    B = random.randint(0, 12)
    A = knapsack([(random.randint(1, 5), random.randint(0, 9)) for _ in range(3)], B)
    C = knapsack([(random.randint(1, 5), random.randint(0, 9)) for _ in range(3)], B)
    assert merge_ans(A, C, B) == merge(A, C, B)
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: merge đúng')

Bài 1: 1 | Bài 2: 3 | Bài 3: merge đúng


In [23]:
# Bài 4: giảm giá khi bất kỳ tổ tiên đã mua
def ancestor_discount(n, present, future, hierarchy, budget):
    B = budget
    children, order = build_tree(n, hierarchy)
    dp = [None] * n
    for u in reversed(order):
        res = []
        for p in (0, 1):
            skip, take = [0] * (B + 1), [0] * (B + 1)
            for c in children[u]:
                skip = merge(skip, dp[c][p], B)     # không mua u: truyền nguyên p
                take = merge(take, dp[c][1], B)     # mua u: mọi hậu duệ có tổ tiên đã mua
            cost = present[u] // 2 if p else present[u]
            row = skip[:]
            for b in range(cost, B + 1):
                row[b] = max(row[b], take[b - cost] + future[u] - cost)
            res.append(row)
        dp[u] = res
    return dp[0][0][B]

random.seed(32)
for _ in range(300):
    case = rand_case(random.randint(1, 9))
    assert ancestor_discount(*case) == wrong_any_ancestor(*case)   # "sai" ở bài gốc = đúng ở biến thể
print('Bài 4: khớp vét cạn của biến thể')

Bài 4: khớp vét cạn của biến thể


In [24]:
# Bài 5: truy vết tập nhân viên được mua
def chosen_set(n, present, future, hierarchy, budget):
    B = budget
    children, order = build_tree(n, hierarchy)
    dp = [None] * n
    for u in reversed(order):
        res = []
        for p in (0, 1):
            skip, take = [0] * (B + 1), [0] * (B + 1)
            for c in children[u]:
                skip = merge(skip, dp[c][0], B)
                take = merge(take, dp[c][1], B)
            cost = present[u] // 2 if p else present[u]
            row = skip[:]
            for b in range(cost, B + 1):
                row[b] = max(row[b], take[b - cost] + future[u] - cost)
            res.append(row)
        dp[u] = res

    def split(tables, b):
        """Chia ngân sách b cho dãy bảng con sao cho tổng = giá trị gộp."""
        prefix = [[0] * (B + 1)]
        for t in tables:
            prefix.append(merge(prefix[-1], t, B))
        parts = []
        for i in range(len(tables), 0, -1):
            for x in range(b + 1):
                if prefix[i - 1][b - x] + tables[i - 1][x] == prefix[i][b]:
                    parts.append(x); b -= x
                    break
        return parts[::-1]

    chosen = []
    def go(u, p, b):
        cost = present[u] // 2 if p else present[u]
        kids = children[u]
        skip_val = split([dp[c][0] for c in kids], b)
        take_ok = b >= cost
        skip_total = sum(dp[c][0][x] for c, x in zip(kids, skip_val))
        if take_ok:
            take_parts = split([dp[c][1] for c in kids], b - cost)
            take_total = sum(dp[c][1][x] for c, x in zip(kids, take_parts)) + future[u] - cost
        if take_ok and take_total > skip_total:
            chosen.append(u + 1)
            for c, x in zip(kids, take_parts):
                go(c, 1, x)
        else:
            for c, x in zip(kids, skip_val):
                go(c, 0, x)
    go(0, 0, B)
    return dp[0][0][B], sorted(chosen)

for case in [examples[2][:5], examples[3][:5]]:
    best, who = chosen_set(*case)
    c, p = cost_profit(case[0], case[1], case[2], case[3], {w - 1 for w in who})
    assert c <= case[4] and p == best
    print('lợi nhuận', best, '| mua:', who, '| chi', c)

lợi nhuận 10 | mua: [1, 3] | chi 8
lợi nhuận 12 | mua: [1, 2, 3] | chi 7


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
dp[u][p] = mảng [0..B] (chi ≤ b), p = cha đã mua?
skip = gộp(dp[c][0] cho mọi con c);  take = gộp(dp[c][1] cho mọi con c)
cost = present[u] // 2 nếu p else present[u]
dp[u][p][b] = max(skip[b], take[b - cost] + future[u] - cost)
đáp án = dp[gốc][0][budget]
gộp(A, C)[b] = max_a A[a] + C[b - a]
```

**Checklist:**

- [ ] Giảm giá chỉ khi **sếp trực tiếp** mua?
- [ ] Hai trạng thái `p` cho mỗi nút?
- [ ] Gộp con bằng **chia ngân sách**, không cộng từng vị trí?
- [ ] Không dùng lợi nhuận để mua thêm?
- [ ] Giá giảm có thể bằng 0 (`1 // 2`)?
- [ ] ID 1-based → 0-based?